# Berry Picker Assistant – Weather features for the yield model
**Team Picky · ML team · TIES4570 CSDP**

Downloads **daily FMI weather (2000–2025)** for berry regions across Finland and turns it into **berry-relevant features per region and year**:
growing degree-days, frost during flowering, summer rain, winter snow, etc.

- Source: Finnish Meteorological Institute open data (WFS, no API key), stored query `fmi::observations::weather::daily::simple`
- Output (in the Output tab): `fmi_daily_raw.csv`, `weather_features_region_year.csv`
- Runs on **CPU** (no GPU needed). Settings: Accelerator = None · Internet = On.

## 1. Regions and weather stations

In [ ]:
import time, requests, xml.etree.ElementTree as ET
import pandas as pd, numpy as np

# Region (as in the Marsi berry-harvest reports)  ->  FMI place names to try (first that works is used)
REGIONS = {
    "Uusimaa":            ["Helsinki-Vantaa", "Vantaa"],
    "Varsinais-Suomi":    ["Turku"],
    "Pirkanmaa":          ["Tampere"],
    "Paijat-Hame":        ["Lahti"],
    "Kymenlaakso":        ["Kouvola", "Kotka"],
    "Etela-Karjala":      ["Lappeenranta"],
    "Etela-Savo":         ["Mikkeli"],
    "Pohjois-Savo":       ["Kuopio"],
    "Pohjois-Karjala":    ["Joensuu", "Ilomantsi"],
    "Keski-Suomi":        ["Jyväskylä"],
    "Etela-Pohjanmaa":    ["Seinäjoki"],
    "Pohjanmaa":          ["Vaasa"],
    "Keski-Pohjanmaa":    ["Kokkola"],
    "Pohjois-Pohjanmaa":  ["Oulu"],
    "Kainuu":             ["Kajaani"],
    "Lappi-south":        ["Rovaniemi"],
    "Lappi-north":        ["Sodankylä", "Ivalo"],
}
YEARS = range(2000, 2026)
URL = "https://opendata.fmi.fi/wfs"

## 2. Download daily weather (one request per station per year)

In [ ]:
def fetch_year(place, year):
    params = {"service": "WFS", "version": "2.0.0", "request": "getFeature",
              "storedquery_id": "fmi::observations::weather::daily::simple",
              "place": place, "starttime": f"{year}-01-01T00:00:00Z", "endtime": f"{year}-12-31T00:00:00Z"}
    for attempt in range(3):
        r = requests.get(URL, params=params, timeout=60)
        if r.status_code == 200:
            break
        time.sleep(5)
    if r.status_code != 200:
        return None
    rows = []
    root = ET.fromstring(r.content)
    for el in root.iter():
        if el.tag.endswith("BsWfsElement"):
            rec = {}
            for ch in el:
                t = ch.tag.split("}")[-1]
                if t in ("Time", "ParameterName", "ParameterValue"):
                    rec[t] = ch.text
            rows.append(rec)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["value"] = pd.to_numeric(df["ParameterValue"], errors="coerce")
    df["date"] = pd.to_datetime(df["Time"]).dt.tz_localize(None).dt.normalize()
    return df.pivot_table(index="date", columns="ParameterName", values="value", aggfunc="mean").reset_index()

frames, used = [], {}
for region, places in REGIONS.items():
    for place in places:
        test = fetch_year(place, 2020)
        if test is not None and len(test) > 300:
            used[region] = place
            break
    if region not in used:
        print("No data for", region); continue
    for y in YEARS:
        d = fetch_year(used[region], y)
        if d is not None and len(d):
            d["region"], d["station"], d["year"] = region, used[region], y
            frames.append(d)
        time.sleep(0.3)
    print(f"{region:20s} <- {used[region]:16s} done")

raw = pd.concat(frames, ignore_index=True)
raw.to_csv("/kaggle/working/fmi_daily_raw.csv", index=False)
print(raw.shape); print(raw.columns.tolist()); raw.head()

## 3. Berry-relevant features per region and year
Why these features (from berry research): bilberry/lingonberry yield depends on **flowering-time frost** (May–June), **pollination weather** (warm, dry days in late May–June), **summer warmth** (degree-days) and **rain in July–August** (berry size), plus **winter snow** protecting the plants.

In [ ]:
raw = pd.read_csv("/kaggle/working/fmi_daily_raw.csv", parse_dates=["date"])
raw["month"], raw["doy"] = raw["date"].dt.month, raw["date"].dt.dayofyear
t, tmin, rr = raw.get("tday"), raw.get("tmin"), raw.get("rrday")
raw["rr"] = raw["rrday"].clip(lower=0) if "rrday" in raw else np.nan   # FMI uses -1 for 'no rain'
raw["gdd5"] = (raw["tday"] - 5).clip(lower=0)

def feats(g):
    m = g["month"]
    out = {
        "gdd5_apr_aug":        g.loc[m.between(4, 8), "gdd5"].sum(),
        "gdd5_may_jun":        g.loc[m.between(5, 6), "gdd5"].sum(),
        "frost_days_flower":   (g.loc[g["doy"].between(130, 175), "tmin"] < -1).sum(),   # ~10 May - 24 Jun
        "min_temp_flower":     g.loc[g["doy"].between(130, 175), "tmin"].min(),
        "warm_days_jun":       (g.loc[m == 6, "tday"] > 12).sum(),
        "rain_may_jun":        g.loc[m.between(5, 6), "rr"].sum(),
        "rain_jul":            g.loc[m == 7, "rr"].sum(),
        "rain_aug":            g.loc[m == 8, "rr"].sum(),
        "dry_days_jul":        (g.loc[m == 7, "rr"] < 0.3).sum(),
        "tmean_jul":           g.loc[m == 7, "tday"].mean(),
        "tmax_jul":            g.loc[m == 7, "tmax"].max() if "tmax" in g else np.nan,
        "snow_max_winter":     g.loc[m.between(1, 3), "snow"].max() if "snow" in g else np.nan,
        "days_with_data":      g["tday"].notna().sum(),
    }
    return pd.Series(out)

F = raw.groupby(["region", "station", "year"]).apply(feats).reset_index()
# Previous-year conditions also matter (flower buds form the summer before)
F = F.sort_values(["region", "year"])
for c in ["gdd5_apr_aug", "rain_jul", "rain_aug"]:
    F[c + "_prev"] = F.groupby("region")[c].shift(1)
F.to_csv("/kaggle/working/weather_features_region_year.csv", index=False)
print(F.shape)
F.describe().T.round(1)

In [ ]:
# Quick sanity look: one region over the years
F[F.region == "Keski-Suomi"][["year", "gdd5_apr_aug", "frost_days_flower", "rain_jul", "rain_aug", "snow_max_winter"]].tail(10)

## ✅ Done
The two CSV files are in the **Output** tab. Next step: join them with the yearly berry-harvest labels (Marsi reports) and train the gradient-boosting model.